# 🚀 DENSO VisionMind — vLLM Qwen2.5-VL Serving with ngrok (Kaggle)

Notebook này phục vụ triển khai mô hình **Qwen2.5-VL-7B-Instruct** (Multimodal Vision) bằng **vLLM Engine** siêu tốc độ cao trên **GPU T4 x 2** của Kaggle, mở cổng API qua **ngrok** để kết nối trực tiếp với Django `PlanAProject`.

### ⚙️ Cài đặt Notebook trên Kaggle (Bắt buộc):
1. Ở bảng **Settings** bên phải màn hình Kaggle:
   - **Accelerator**: Chọn **GPU T4 x 2** (32GB VRAM gộp).
   - **Internet**: Bật **ON** (để tải mô hình và mở tunnel ngrok).
2. Đăng nhập [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) để lấy **Authtoken** miễn phí.

In [ ]:
# 1. Cài đặt vLLM Engine và pyngrok
!pip install -q -U vllm pyngrok
print("✅ Cài đặt vLLM và pyngrok hoàn tất!")

In [ ]:
from pyngrok import ngrok

# 2. Cấu hình ngrok Authtoken
# 🔴 Dán token của bạn từ https://dashboard.ngrok.com/get-started/your-authtoken vào đây:
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ Đã cấu hình ngrok token thành công!")

In [ ]:
import subprocess
import time

# 3. Khởi động vLLM Server phục vụ Qwen2.5-VL trên cụm 2 GPU T4
# Nếu bạn dùng checkpoint riêng đã train: thay 'Qwen/Qwen2.5-VL-7B-Instruct' bằng đường dẫn model của bạn
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

cmd = f"""
python3 -m vllm.entrypoints.openai.api_server \
    --model {MODEL_NAME} \
    --port 8000 \
    --tensor-parallel-size 2 \
    --gpu-memory-utilization 0.90 \
    --max-model-len 4096 \
    --trust-remote-code \
    --dtype bfloat16
"""

print(f"🚀 Đang khởi động vLLM Server với model: {MODEL_NAME}...")
print("⏳ Quá trình tải weights và tối ưu hóa VRAM mất khoảng 60-90 giây, vui lòng chờ...")
vllm_proc = subprocess.Popen(cmd, shell=True)

# Chờ 60 giây để vLLM nạp mô hình vào 2 GPU T4
time.sleep(60)
print("✅ Tiến trình vLLM Server đã được kích hoạt ngầm!")

In [ ]:
import requests
from pyngrok import ngrok

# 4. Kiểm tra sức khỏe của Server nội bộ
ready = False
print("⏳ Đang kiểm tra cổng nội bộ 127.0.0.1:8000...")
for i in range(15):
    try:
        res = requests.get("http://127.0.0.1:8000/v1/models", timeout=3)
        if res.status_code == 200:
            print(f"✅ Server đã sẵn sàng! Danh sách model: {res.json()}")
            ready = True
            break
    except Exception:
        time.sleep(5)

if not ready:
    print("⚠️ Server vẫn đang nạp weights, bạn có thể chạy lại cell này sau 15-30 giây.")
else:
    # Đóng tunnel cũ nếu có và mở tunnel ngrok mới
    ngrok.kill()
    http_tunnel = ngrok.connect(8000)
    public_url = http_tunnel.public_url

    print("\n" + "=" * 70)
    print("🎉 ĐƯỜNG LINK NGROK ĐÃ SẴN SÀNG CHO PLAN A PROJECT!")
    print(f"👉 VLLM_BASE_URL={public_url}/v1")
    print(f"👉 VLLM_MODEL={MODEL_NAME}")
    print(f"👉 VLLM_API_KEY=EMPTY")
    print("=" * 70)
    print("\n📋 HÃY COPY 3 DÒNG TRÊN VÀO FILE 'PlanAProject/.env' TRÊN MÁY TÍNH CỦA BẠN!")

In [ ]:
# 5. Test thử một lượt truy vấn OpenAI-compatible ngay trên Kaggle
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="EMPTY")
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Chào bạn! Bạn có thể hỗ trợ phân tích bản vẽ cơ khí và sơ đồ robot DENSO không?"}
    ],
    max_tokens=100
)

print("💬 Phản hồi thử nghiệm từ vLLM:")
print(response.choices[0].message.content)